In [ ]:
import os
from pathlib import Path

import pandas as pd


In [ ]:
path = Path.cwd()
data_path = path.parent / "data"

### load the dataset

In [ ]:
df_raw = pd.read_csv(data_path / "csic_2010_raw.csv", low_memory=False)
df_raw

**Observations:**  

The raw CSV contains:

- The raw dataset contains approximately 223,585 parameter level rows and 18 columns.
- These rows may represent logical HTTP requests grouped by `index`.  
- The `method` represents HTTP methods, such as GET, POST, or PUT. Can be used a categorial column
- The `url` column Request URL or URI. Can be featurised in multiple ways based on:
  - Counts on special chars in url such as '&', '/', '.', '=' etc
  - Count of query parameters
  - Length of `url`
  - Presence of well know suspicious parterns such as:
    - sql: select, union, drop, insert, delete
    - html: script tag
    - Path traversal pattern
- The `protocol` column represents HTTP protocol version. Seems to be a constant value column
- The `user_agent`column represents User-Agent header value. Seems to be a constant value column
- HTTP header field columns `pragma`, `cache_control`, `accept`, `accept_encoding`, `accept_charset`, `accept_language`, `host` and `connection` seem to be constant value columns  
- The `content_length` contains a numeric content length value
- The `contentType`  can be used a categorial column
- The `payload` columns represent the request payload. Can be featurised in multiple ways based on: 
    - Whether payload is present or not
    - Length of `payload`
    - Counts on special chars in payload such as '&', '/', '.', '=' etc
    - Presence of well know suspicious parterns such as:
        - sql: select, union, drop, insert, delete
        - html: script tag
        - Path traversal pattern
- The `label` is the target column with `anom` (anomalous) and `norm` (normal) values

### Check the data types and missing values in the various columns:

In [ ]:
df_raw.info()

**Observations:**  
- `contentLength`, `contentType`, `payload` colums have values missing.
- The target `label` has no values missing i.e no rows to discard

### Analyse the uniqueness of each column

In [ ]:
print(f"index: {df_raw['index'].nunique()}")
print(f"method: {df_raw['method'].nunique()}")
print(f"url: {df_raw['url'].nunique()}")
print(f"protocol: {df_raw['protocol'].nunique()}")
print(f"userAgent: {df_raw['userAgent'].nunique()}")
print(f"pragma: {df_raw['pragma'].nunique()}")
print(f"cacheControl: {df_raw['cacheControl'].nunique()}")
print(f"accept: {df_raw['accept'].nunique()}")
print(f"acceptEncoding: {df_raw['acceptEncoding'].nunique()}")
print(f"acceptCharset: {df_raw['acceptCharset'].nunique()}")
print(f"acceptLanguage: {df_raw['acceptLanguage'].nunique()}")
print(f"host: {df_raw['host'].nunique()}")
print(f"connection: {df_raw['connection'].nunique()}")
print(f"contentLength: {df_raw['contentLength'].nunique()}")
print(f"contentType: {df_raw['contentType'].nunique()}")
print(f"cookie: {df_raw['cookie'].nunique()}")
print(f"payload: {df_raw['payload'].nunique()}")
print(f"label: {df_raw['label'].nunique()}")

**Observations:**  
- `index` column shows `36000` unique values whereas total rows count is `223585`, It implies that index column values are not unique. 
- `method` column with `3` unique values (GET, POST, PUT), Can be used as Categorial feature (one-hot encode)
- `url` with `1643` unique values. Various features columns can be derived from iteg: url_length, count of special characters, presense of malicious patterns etc
- `host` column has `2` unique values. Can be used as Categorical feature (one-hot encode)
- `contentLength` column has `282` unique values. Can be used as Numeric feature  column.
- `cookie` column has `61065` unique values. Can be used as Numeric feature  column.
- `payload` column with `33769` unique values. Various features columns can be derived from it eg: payload_length, count of special characters, presense of malicious patterns etc
- `protocol`, `userAgent`, `pragma`, `cacheControl`, `accept`, `acceptEncoding`, `acceptCharset`, `acceptLanguage`, `connection`, `contentType` all these columns have a single unique value. These columns can be dropped.
- `label` column has `2` values. Can be used as Categorical feature 

### Drop Constant Columns:

In [ ]:
const_cols = ["protocol", "userAgent", "pragma", "cacheControl", "accept", "acceptEncoding", "acceptCharset", "acceptLanguage", "connection", "contentType"]

df_multi = df_raw.drop(columns = const_cols)
df_multi

### Analyze rows for `index=0`

In [ ]:
group_index0 = df_multi[df_raw['index'] == 0]
group_index0

**Observations:**  

- HTTP Request 1 (Anomalous):  
  - Rows: `0` to `0`  
  - Cookie: `JSESSIONID=B92A8B48B9008CD29F622A994E0F650D`  
  - Label: `anom`  
  - Payload: Fragmented `id=2`, `nombre=Jam%F3n+Ib%E9rico`, `precio=85`, `cantidad=%27%3B+DROP+TABLE+usuarios%3B+SELECT+*+FROM+datos+WHERE+nombre+LIKE+%27%25`, `B1=A%F1adir+al+carrito`
- HTTP Request 2 (Normal):  
  - Rows: 119585  
  - Cookie: `JSESSIONID=1F767F17239C9B670A39E9B10C3825F4`  
  - Label: `norm`  
  - Payload: `NaN`  
- `index=0` is reused for both requests.  
- `cookie` uniquely distinguishes both requests.  
- `Payload` is split across multiple rows for the for HTTP Request 1. Combine fragments to reconstruct the full payload.  
- All rows in a HTTP request share the same `label` (`anom` or `norm`).  

### Analyze rows for `index=1`

In [ ]:
group_index1 = df_multi[df_raw['index'] == 1]
group_index1

**Observations:**  

- HTTP Request 1 (Anomalous):  
  - Rows: `5` to `9`  
  - Cookie: `JSESSIONID=F563B5262843F12ECAE41815ABDEEA54`  
  - Label: `anom`  
  - Payload: Fragmented `id=2%2F`, `nombre=Jamón`, `precio=85`, `cantidad=49`, `B1=A%F1adir+al+carrito`
- HTTP Request 2 (Normal):  
  - Rows: 119586–119590  
  - Cookie: `JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5`  
  - Label: `norm`  
  - Payload: Fragmented `id=3`, `nombre=Vino+Rioja`, `precio=100`, `cantidad=55`, `B1=A%F1adir+al+carrito`  
- `index=1` is reused for both requests.  
- `cookie` uniquely distinguishes both requests.  
- `Payloads` are split across multiple rows for the same request. Combine fragments to reconstruct the full payload.  
- All rows in a HTTP request share the same `label` (`anom` or `norm`).  

> **Observations:**  
> - `10` rows in total for `index=1`
> - Rows# `5` to `9` and `119586` to `119590` implies non-contiguous row distribution for `index=1`

**Based on the analysis of `index=0` and `index=1` following conclusions can be derived:**  
1. `cookie` is a Unique Identifier for HTTP Requests  
2. `index` column is not unique
3. `Payload` column fragments can be reassembled using cookie
4. All Rows in a HTTP Request share the same `label`

### Check columns have consistent/same value for a specific `cookie`

In [ ]:
# Dictionary to store inconsistencies
inconsistencies = {
    'index': [],
    'method': [],
    'url': [],
    'host': [],
    'label': [],
    'contentLength': []
}

# Group by Cookie
grouped = df_multi.groupby('cookie')

# Check for inconsistencies in each column
for name, group in grouped:
    for col in inconsistencies.keys():
        unique_values = group[col].unique()
        if len(unique_values) != 1:
            inconsistencies[col].append([name, unique_values])

# Print the number of inconsistencies for each column
for col, inconsistencies_list in inconsistencies.items():
    print(f"Cookies with inconsistent {col}s: {len(inconsistencies_list)}")


- All columns are consistent for each cookie group.
- No groups have mixed values for `index`, `method`, `url`, `host`, `label`, or `contentLength`.


In [ ]:
df_reconstructed = (
    grouped.agg({
        'index': 'first',  # Take the first index (all rows in the group share the same index)
        'method': 'first',  # Take the first method
        'url': 'first',     # Take the first URL
        'host': 'first',     # Take the first host
        'label': 'first',   # Take the first label (all rows in the group share the same label)
        'payload': lambda x: ' '.join(x.dropna()),  # Combine all non-NaN payload fragments
        'contentLength': 'first',
        'number_of_payload_fragments': 'size',  # Count the number of fragments
    })
    .reset_index()  # Convert 'cookie' from index to a column
)

df_reconstructed

In [ ]:
df_multi['contentLength'][df_multi['index'] == 19522]